# 🌍 City Quality of Life — Dataset Merge

Merges city-level QoL datasets into a single unified dataframe.

**Pipeline overview:**

| Step | What happens |
|---|---|
| 1 | Load & inspect source datasets |
| 2 | Normalise city/country names (strip suffixes, unify separators) |
| 3 | Merge QoL core datasets (`qol_index_2024` + `qol_dataset`) — overlapping concepts kept with source suffix |
| 4 | Merge salary — last non-null year from `net_salary` |
| 5 | Modular enrichment from the big cost-of-living dataset (user-selectable columns) |

## 1. Setup

In [1]:
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz

warnings.filterwarnings("ignore")

DATA_DIR = Path("data/raw")

# ── Fuzzy-match settings ──────────────────────────────────────────────────────
FUZZY_THRESHOLD = 88   # minimum similarity score (0-100) to accept a match
FUZZY_SCORER   = fuzz.WRatio   # handles partial matches, transpositions, etc.

print("✅ Setup complete")

✅ Setup complete


## 2. Name-normalisation helpers

All city/country normalisation goes through a single pipeline so every dataset
is pre-processed identically before any join.

In [2]:
# Common country name variants that appear after a comma in city strings
_COUNTRY_ALIASES = {
    "usa": "united states", "us": "united states", "u.s.": "united states",
    "uk": "united kingdom", "u.k.": "united kingdom",
    "uae": "united arab emirates",
}


def _strip_accents(s: str) -> str:
    """Decompose unicode and drop combining marks (accents)."""
    return "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )


def normalise_city(raw: str) -> str:
    """
    Return a canonical city name suitable for fuzzy matching.

    Transformations applied (in order):
      1. Strip leading/trailing whitespace
      2. Split on " / " or "," and keep the FIRST token  
         → "Rome, Italy" → "Rome"
         → "New York / Manhattan" → "New York"
      3. Remove parenthetical suffixes  → "Washington (D.C.)" → "Washington"
      4. Remove trailing state/region after " - "  → "Austin - TX" → "Austin"
      5. Lowercase + strip accents
      6. Collapse multiple spaces
    """
    if not isinstance(raw, str):
        return ""
    s = raw.strip()
    # split on comma or slash, keep first part
    s = re.split(r"[,/]", s)[0].strip()
    # remove parenthetical suffixes
    s = re.sub(r"\s*\(.*?\)", "", s).strip()
    # remove trailing " - region"
    s = re.sub(r"\s*-\s*\w{2,}$", "", s).strip()
    s = _strip_accents(s.lower())
    s = re.sub(r"\s+", " ", s)
    return s


def normalise_country(raw: str) -> str:
    """Lowercase + strip accents + resolve common aliases."""
    if not isinstance(raw, str):
        return ""
    s = _strip_accents(raw.strip().lower())
    return _COUNTRY_ALIASES.get(s, s)


def add_norm_keys(df: pd.DataFrame,
                  city_col: str,
                  country_col: str | None = None) -> pd.DataFrame:
    """Add `_city_key` (and optionally `_country_key`) normalised columns."""
    df = df.copy()
    df["_city_key"] = df[city_col].map(normalise_city)
    if country_col:
        df["_country_key"] = df[country_col].map(normalise_country)
    return df


print("✅ Normalisation helpers defined")

# Quick smoke-test
_tests = [
    ("Rome, Italy",              "rome"),
    ("New York / Manhattan",     "new york"),
    ("São Paulo",                "sao paulo"),
    ("Washington (D.C.)",        "washington"),
    ("Austin - TX",              "austin"),
    ("  Luxembourg  ",           "luxembourg"),
]
for raw, expected in _tests:
    result = normalise_city(raw)
    status = "✅" if result == expected else "❌"
    print(f"  {status}  {repr(raw):35s} → {repr(result):25s}  (expected {repr(expected)})")

✅ Normalisation helpers defined
  ✅  'Rome, Italy'                       → 'rome'                     (expected 'rome')
  ✅  'New York / Manhattan'              → 'new york'                 (expected 'new york')
  ✅  'São Paulo'                         → 'sao paulo'                (expected 'sao paulo')
  ✅  'Washington (D.C.)'                 → 'washington'               (expected 'washington')
  ✅  'Austin - TX'                       → 'austin'                   (expected 'austin')
  ✅  '  Luxembourg  '                    → 'luxembourg'               (expected 'luxembourg')


## 3. Load QoL datasets

In [3]:
# ── qol_index_2024  (150 cities, Numbeo composite scores) ─────────────────────
qol_index = pd.read_csv(DATA_DIR / "qol_index_2024" / "livable_cities.csv")

# Rename to clean snake_case; keep originals for reference
qol_index = qol_index.rename(columns={
    "City":                            "city",
    "Country":                         "country",
    "Quality of Life Index":           "qol_index_numbeo",
    "Purchasing Power Index":          "purchasing_power_idx_numbeo",
    "Safety Index":                    "safety_idx_numbeo",
    "Health Care Index":               "healthcare_idx_numbeo",
    "Cost of Living Index":            "cost_of_living_idx_numbeo",
    "Property Price to Income Ratio":  "property_price_to_income_numbeo",
    "Traffic Commute Time Index":      "traffic_commute_idx_numbeo",
    "Pollution Index":                 "pollution_idx_numbeo",
    "Climate Index":                   "climate_idx_numbeo",
})
qol_index = qol_index.drop(columns=["Rank"], errors="ignore")
qol_index = add_norm_keys(qol_index, "city", "country")

print(f"qol_index_2024   : {qol_index.shape[0]} cities × {qol_index.shape[1]} cols")
display(qol_index.head(3))

qol_index_2024   : 150 cities × 13 cols


,city,country,qol_index_numbeo,purchasing_power_idx_numbeo,safety_idx_numbeo,healthcare_idx_numbeo,cost_of_living_idx_numbeo,property_price_to_income_numbeo,traffic_commute_idx_numbeo,pollution_idx_numbeo,climate_idx_numbeo,_city_key,_country_key
0,The Hague,Netherlands,223.8,138.9,79.8,80.7,60.2,5.9,21.0,17.5,90.6,the hague,netherlands
1,Luxembourg,Luxembourg,220.6,175.9,71.4,76.6,62.0,9.2,27.7,21.6,82.6,luxembourg,luxembourg
2,Eindhoven,Netherlands,217.5,139.4,78.1,78.9,62.2,6.0,24.5,19.2,85.4,eindhoven,netherlands


In [4]:
# ── qol_dataset  (266 cities, Teleport dimensional scores 0-10) ───────────────
qol_tel = pd.read_csv(DATA_DIR / "qol_dataset" / "uaScoresDataFrame.csv")

qol_tel = qol_tel.drop(columns=["Unnamed: 0"], errors="ignore")
qol_tel = qol_tel.rename(columns={
    "UA_Name":      "city",
    "UA_Country":   "country",
    "UA_Continent": "continent",
})

# Suffix ALL score columns with _teleport to distinguish from Numbeo
score_cols = [c for c in qol_tel.columns if c not in ("city", "country", "continent")]
qol_tel = qol_tel.rename(columns={c: c.lower().replace(" ", "_").replace("&", "and") + "_teleport"
                                   for c in score_cols})
qol_tel = add_norm_keys(qol_tel, "city", "country")

print(f"qol_dataset      : {qol_tel.shape[0]} cities × {qol_tel.shape[1]} cols")
print(f"Teleport columns : {[c for c in qol_tel.columns if c.endswith('_teleport')]}")
display(qol_tel.head(3))

qol_dataset      : 266 cities × 22 cols
Teleport columns : ['housing_teleport', 'cost_of_living_teleport', 'startups_teleport', 'venture_capital_teleport', 'travel_connectivity_teleport', 'commute_teleport', 'business_freedom_teleport', 'safety_teleport', 'healthcare_teleport', 'education_teleport', 'environmental_quality_teleport', 'economy_teleport', 'taxation_teleport', 'internet_access_teleport', 'leisure_and_culture_teleport', 'tolerance_teleport', 'outdoors_teleport']


,city,country,continent,housing_teleport,cost_of_living_teleport,startups_teleport,venture_capital_teleport,travel_connectivity_teleport,commute_teleport,business_freedom_teleport,...,education_teleport,environmental_quality_teleport,economy_teleport,taxation_teleport,internet_access_teleport,leisure_and_culture_teleport,tolerance_teleport,outdoors_teleport,_city_key,_country_key
0,Aarhus,Denmark,Europe,6.1315,4.015,2.8270,2.512,3.5360,6.31175,9.940000,...,5.3665,7.63300,4.8865,5.0680,8.373,3.1870,9.7385,4.1300,aarhus,denmark
1,Adelaide,Australia,Oceania,6.3095,4.692,3.1365,2.640,1.7765,5.33625,9.399667,...,5.1420,8.33075,6.0695,4.5885,4.341,4.3285,7.8220,5.5310,adelaide,australia
2,Albuquerque,New Mexico,North America,7.2620,6.059,3.7720,1.493,1.4555,5.05575,8.671000,...,4.1520,7.31950,6.5145,4.3460,5.396,4.8900,7.0285,3.5155,albuquerque,new mexico


## 4. Fuzzy merge — QoL datasets

Strategy:
* Union of all unique cities across both datasets (maximise coverage).
* For each city in the smaller `qol_index` (150), find the best fuzzy match
  in `qol_tel` (266).  If score ≥ threshold **and** countries agree, merge.
* Cities that appear only in one dataset get NaN for the other's columns.

In [5]:
def fuzzy_merge(left: pd.DataFrame,
                right: pd.DataFrame,
                left_key: str  = "_city_key",
                right_key: str = "_city_key",
                left_country:  str | None = "_country_key",
                right_country: str | None = "_country_key",
                threshold: int = FUZZY_THRESHOLD,
                scorer = FUZZY_SCORER) -> pd.DataFrame:
    """
    Outer-join `left` and `right` using fuzzy city-name matching.

    Returns a dataframe with all rows from both sides. Matched pairs share
    a row; unmatched rows appear once with NaN for the opposite side's columns.
    A `_match_score` column records the similarity score (NaN = no match found).
    """
    right_keys = right[right_key].tolist()

    matched_left_idx  = []
    matched_right_idx = []
    scores            = []

    for li, lrow in left.iterrows():
        lk = lrow[left_key]
        if not lk:
            continue

        result = process.extractOne(lk, right_keys, scorer=scorer,
                                    score_cutoff=threshold)
        if result is None:
            continue

        best_name, score, best_pos = result
        ri = right.index[best_pos]

        # Optional country guard: if both sides have a country, they must agree
        if left_country and right_country:
            lc = lrow.get(left_country, "")
            rc = right.at[ri, right_country]
            if lc and rc and lc != rc:
                continue   # same city name, different country → skip

        matched_left_idx.append(li)
        matched_right_idx.append(ri)
        scores.append(score)

    # Build matched pairs
    left_matched  = left.loc[matched_left_idx].reset_index(drop=True)
    right_matched = right.loc[matched_right_idx].reset_index(drop=True)

    # Drop duplicate key/country cols from right before concat
    drop_from_right = [c for c in [right_key, right_country, "city", "country"]
                       if c and c in right_matched.columns]
    right_matched = right_matched.drop(columns=drop_from_right, errors="ignore")

    merged_matched = pd.concat([left_matched, right_matched], axis=1)
    merged_matched["_match_score"] = scores

    # Unmatched left rows
    unmatched_left = left.drop(index=matched_left_idx)

    # Unmatched right rows (drop those already matched)
    unmatched_right = right.drop(index=matched_right_idx)
    drop_from_right2 = [c for c in [right_key, right_country]
                        if c and c in unmatched_right.columns]
    unmatched_right = unmatched_right.drop(columns=drop_from_right2, errors="ignore")
    # Rename right city/country so we can coalesce below
    unmatched_right = unmatched_right.rename(columns={"city": "city", "country": "country"})

    result = pd.concat([merged_matched, unmatched_left, unmatched_right],
                       ignore_index=True, sort=False)
    return result


print("✅ fuzzy_merge helper defined")

✅ fuzzy_merge helper defined


In [6]:
# ── Run the QoL merge ─────────────────────────────────────────────────────────
qol_merged = fuzzy_merge(qol_index, qol_tel)

# Coalesce city / country: prefer qol_index value, fall back to qol_tel value
# (both already present as 'city' / 'country' from left and right)
# continent only comes from qol_tel
qol_merged["city"]    = qol_merged["city"].fillna("")
qol_merged["country"] = qol_merged["country"].fillna("")

# Canonical city/country: first non-empty value across the two sources
# (after fuzzy_merge the left columns are already the primary)
# Clean up internal key columns
qol_merged = qol_merged.drop(columns=["_city_key", "_country_key"], errors="ignore")

# Sort for readability
qol_merged = qol_merged.sort_values("city").reset_index(drop=True)

matched   = qol_merged["_match_score"].notna().sum()
total     = len(qol_merged)
print(f"\n📊 Merge result")
print(f"   Total cities    : {total}")
print(f"   Matched pairs   : {matched}")
print(f"   Only in index   : {(qol_merged['_match_score'].isna() & qol_merged['qol_index_numbeo'].notna()).sum()}")
print(f"   Only in teleport: {(qol_merged['_match_score'].isna() & qol_merged['housing_teleport'].notna()).sum()}")
print(f"   Columns         : {qol_merged.shape[1]}")
display(qol_merged.head(5))


📊 Merge result
   Total cities    : 327
   Matched pairs   : 89
   Only in index   : 61
   Only in teleport: 177
   Columns         : 30


,city,country,qol_index_numbeo,purchasing_power_idx_numbeo,safety_idx_numbeo,healthcare_idx_numbeo,cost_of_living_idx_numbeo,property_price_to_income_numbeo,traffic_commute_idx_numbeo,pollution_idx_numbeo,...,healthcare_teleport,education_teleport,environmental_quality_teleport,economy_teleport,taxation_teleport,internet_access_teleport,leisure_and_culture_teleport,tolerance_teleport,outdoors_teleport,_match_score
0,Aarhus,Denmark,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,8.704333,5.3665,7.63300,4.8865,5.0680,8.373,3.1870,9.7385,4.1300,NaN
1,Abu Dhabi,UnitedArabEmirates,192.4,160.5,88.2,71.1,54.1,5.8,32.9,46.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Adelaide,Australia,189.4,109.3,67.7,73.9,76.0,7.9,35.5,24.3,...,7.936667,5.1420,8.33075,6.0695,4.5885,4.341,4.3285,7.8220,5.5310,100.0
3,Ahmedabad,India,124.5,63.4,67.9,67.2,22.8,7.9,39.0,72.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Albuquerque,New Mexico,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.430000,4.1520,7.31950,6.5145,4.3460,5.396,4.8900,7.0285,3.5155,NaN


In [7]:
# ── Inspect low-confidence matches ───────────────────────────────────────────
low = qol_merged[qol_merged["_match_score"].notna() &
                 (qol_merged["_match_score"] < 95)]
print(f"⚠️  {len(low)} matches below 95 score — review:")
display(low[["city", "country", "_match_score"]].sort_values("_match_score"))

⚠️  1 matches below 95 score — review:


,city,country,_match_score
224,Panama City,Panama,90.0


## 5. Merge salary — last non-null year

In [8]:
salary_raw = pd.read_csv(
    DATA_DIR / "net_salary" /
    "cities_average_net_salary_after_tax.19-10-2021.csv"
)

# ── Fix messy column names ────────────────────────────────────────────────────
# Original columns have embedded quotes and mixed patterns like:
#   'City', ' "Region"', ' "Country"',
#   ' "Average Monthly Net Salary (After Tax)', ' 2010"', ...
# The year columns come in *pairs*: the first col of each pair holds
# the value, the second is just the year string with a closing quote.
# We reconstruct clean year labels by pairing them up.

raw_cols = salary_raw.columns.tolist()

clean_cols = ["city", "region", "country"]
year_pairs = raw_cols[3:]   # everything after city/region/country

# They come as: [salary_col_2010, '2010"'], [salary_col_2011, '2011"'], ...
# Extract the 4-digit year from the second element of each pair
salary_year_cols = {}
for i in range(0, len(year_pairs), 2):
    raw_year_tag = year_pairs[i + 1] if i + 1 < len(year_pairs) else ""
    year = re.search(r"(\d{4})", str(raw_year_tag))
    year_label = int(year.group(1)) if year else 2010 + i // 2
    salary_year_cols[year_pairs[i]] = f"salary_{year_label}"

rename_map = {raw_cols[0]: "city",
              raw_cols[1]: "region",
              raw_cols[2]: "country"}
rename_map.update(salary_year_cols)
# mark the pure-year label columns for dropping
drop_year_label_cols = [year_pairs[i+1] for i in range(0, len(year_pairs), 2)
                        if i + 1 < len(year_pairs)]

salary = salary_raw.rename(columns=rename_map)
salary = salary.drop(columns=drop_year_label_cols, errors="ignore")

# Strip leading/trailing quotes from city/country text
for col in ["city", "region", "country"]:
    salary[col] = salary[col].astype(str).str.strip().str.strip('"')

year_cols = [c for c in salary.columns if c.startswith("salary_")]
salary[year_cols] = salary[year_cols].apply(pd.to_numeric, errors="coerce")

print(f"Salary dataset: {salary.shape}")
print(f"Year columns  : {year_cols}")
display(salary.head(3))

Salary dataset: (685, 14)
Year columns  : ['salary_2010', 'salary_2011', 'salary_2012', 'salary_2013', 'salary_2014', 'salary_2015', 'salary_2016', 'salary_2017', 'salary_2018', 'salary_2019', 'salary_2020']


,city,region,country,salary_2010,salary_2011,salary_2012,salary_2013,salary_2014,salary_2015,salary_2016,salary_2017,salary_2018,salary_2019,salary_2020
0,New York City,New York,United States of America,3300.0,4017.991667,3476.340323,4026.656605,4658.585654,6023.253437,NaN,NaN,NaN,NaN,NaN
1,"Washington, D.C.",District of Columbia,United States of America,2200.0,3700.000000,4573.356667,3900.222222,4762.834321,5601.921903,NaN,NaN,NaN,NaN,NaN
2,San Francisco,California,United States of America,4700.0,4195.428571,3470.000000,4281.510400,6111.775926,7930.791453,NaN,NaN,NaN,NaN,NaN


In [9]:
# ── Pick the most recent non-null salary per city ────────────────────────────
def last_non_null(row: pd.Series, cols: list[str]) -> tuple[float, int | None]:
    """Return (value, year) of the last non-NaN entry across `cols`."""
    for col in reversed(cols):   # reversed = most recent first
        if pd.notna(row[col]):
            year = int(col.split("_")[1])
            return row[col], year
    return np.nan, None


salary[["salary_net_usd", "salary_year"]] = salary.apply(
    lambda r: pd.Series(last_non_null(r, year_cols)),
    axis=1
)
salary["salary_year"] = salary["salary_year"].astype("Int64")  # nullable int

# Keep only what we need for the join
salary_slim = salary[["city", "country", "salary_net_usd", "salary_year"]].copy()
salary_slim = salary_slim[salary_slim["salary_net_usd"].notna()]
salary_slim = add_norm_keys(salary_slim, "city", "country")

print(f"Salary rows with data: {len(salary_slim)}")
print(f"Year distribution:")
display(salary_slim["salary_year"].value_counts().sort_index())

Salary rows with data: 685
Year distribution:


salary_year
2015    685
Name: count, dtype: Int64

In [10]:
# ── Fuzzy-join salary onto the merged QoL dataset ────────────────────────────
# The QoL dataset already has city / country; add norm keys back for the join
qol_merged = add_norm_keys(qol_merged, "city", "country")

right_keys_sal = salary_slim["_city_key"].tolist()

sal_values, sal_years, sal_scores = [], [], []

for _, row in qol_merged.iterrows():
    lk = row["_city_key"]
    lc = row["_country_key"]
    if not lk:
        sal_values.append(np.nan); sal_years.append(pd.NA); sal_scores.append(np.nan)
        continue

    res = process.extractOne(lk, right_keys_sal, scorer=FUZZY_SCORER,
                             score_cutoff=FUZZY_THRESHOLD)
    if res is None:
        sal_values.append(np.nan); sal_years.append(pd.NA); sal_scores.append(np.nan)
        continue

    _, score, pos = res
    matched_row = salary_slim.iloc[pos]

    # Country guard
    rc = matched_row["_country_key"]
    if lc and rc and lc != rc:
        sal_values.append(np.nan); sal_years.append(pd.NA); sal_scores.append(np.nan)
        continue

    sal_values.append(matched_row["salary_net_usd"])
    sal_years.append(matched_row["salary_year"])
    sal_scores.append(score)

qol_merged["salary_net_usd"]  = sal_values
qol_merged["salary_year"]     = pd.array(sal_years, dtype="Int64")
qol_merged["_salary_match_score"] = sal_scores

filled = qol_merged["salary_net_usd"].notna().sum()
print(f"✅ Salary merged: {filled}/{len(qol_merged)} cities matched")

✅ Salary merged: 190/327 cities matched


In [11]:
# ── Finalise base dataset ─────────────────────────────────────────────────────
# Drop internal join keys; keep _match_score cols for traceability
base = qol_merged.drop(columns=["_city_key", "_country_key"], errors="ignore")

# Move city / country / continent to front
front = ["city", "country", "continent"]
rest  = [c for c in base.columns if c not in front]
base  = base[front + rest]

base = base.sort_values("city").reset_index(drop=True)

print(f"\n🗂️  Base dataset: {base.shape[0]} cities × {base.shape[1]} columns")
print(f"   Columns: {list(base.columns)}")
display(base.head(5))


🗂️  Base dataset: 327 cities × 33 columns
   Columns: ['city', 'country', 'continent', 'qol_index_numbeo', 'purchasing_power_idx_numbeo', 'safety_idx_numbeo', 'healthcare_idx_numbeo', 'cost_of_living_idx_numbeo', 'property_price_to_income_numbeo', 'traffic_commute_idx_numbeo', 'pollution_idx_numbeo', 'climate_idx_numbeo', 'housing_teleport', 'cost_of_living_teleport', 'startups_teleport', 'venture_capital_teleport', 'travel_connectivity_teleport', 'commute_teleport', 'business_freedom_teleport', 'safety_teleport', 'healthcare_teleport', 'education_teleport', 'environmental_quality_teleport', 'economy_teleport', 'taxation_teleport', 'internet_access_teleport', 'leisure_and_culture_teleport', 'tolerance_teleport', 'outdoors_teleport', '_match_score', 'salary_net_usd', 'salary_year', '_salary_match_score']


,city,country,continent,qol_index_numbeo,purchasing_power_idx_numbeo,safety_idx_numbeo,healthcare_idx_numbeo,cost_of_living_idx_numbeo,property_price_to_income_numbeo,traffic_commute_idx_numbeo,...,economy_teleport,taxation_teleport,internet_access_teleport,leisure_and_culture_teleport,tolerance_teleport,outdoors_teleport,_match_score,salary_net_usd,salary_year,_salary_match_score
0,Aarhus,Denmark,Europe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,4.8865,5.0680,8.373,3.1870,9.7385,4.1300,NaN,NaN,<NA>,NaN
1,Abu Dhabi,UnitedArabEmirates,NaN,192.4,160.5,88.2,71.1,54.1,5.8,32.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN
2,Adelaide,Australia,Oceania,189.4,109.3,67.7,73.9,76.0,7.9,35.5,...,6.0695,4.5885,4.341,4.3285,7.8220,5.5310,100.0,3075.384286,2015,100.0
3,Ahmedabad,India,NaN,124.5,63.4,67.9,67.2,22.8,7.9,39.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,355.195435,2015,100.0
4,Albuquerque,New Mexico,North America,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.5145,4.3460,5.396,4.8900,7.0285,3.5155,NaN,NaN,<NA>,NaN


## 6. Modular enrichment from the big cost-of-living dataset

**How to use:**
Edit `COLUMNS_TO_ADD` below — it accepts `x`-codes (e.g. `"x48"`) or
human-readable aliases (see `COL_CATALOGUE`).  
Re-run cells 6.1 → 6.3 to regenerate the enriched dataset.

The default `["x2"]` adds the mid-range restaurant meal price for 2.

In [17]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  USER CONFIGURATION — edit this list, then re-run from here downward   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

COLUMNS_TO_ADD: list[str] = [
    "x3",    # Meal for 2, mid-range restaurant (default)
    # Add any of the codes below, or their aliases:
    # "x1",  "x2",  "x4",  "x5",  "x6",  "x7",  "x8",   # restaurants
    # "x9" – "x27",                                        # groceries
    # "x28" – "x35",                                       # transport
    # "x36", "x37", "x38",                                 # utilities
    # "x39", "x40", "x41",                                 # sports/leisure
    # "x42", "x43",                                        # education
    # "x44" – "x47",                                       # clothing
    # "x48", "x49", "x50", "x51",                          # rent
    # "x52", "x53",                                        # property price/m²
    # "x54",                                               # avg net salary
    # "x55",                                               # mortgage rate
]

In [18]:
# ── 6.1  Column catalogue (x-code → human label) ─────────────────────────────
COL_CATALOGUE: dict[str, str] = {
    # Restaurants
    "x1":  "meal_inexpensive_restaurant_usd",
    "x2":  "meal_midrange_2pax_usd",
    "x3":  "mcmeal_combo_usd",
    "x4":  "domestic_beer_draught_usd",
    "x5":  "imported_beer_bottle_usd",
    "x6":  "cappuccino_usd",
    "x7":  "coke_pepsi_usd",
    "x8":  "water_bottle_033l_usd",
    # Groceries
    "x9":  "milk_1l_usd",
    "x10": "bread_500g_usd",
    "x11": "rice_1kg_usd",
    "x12": "eggs_12_usd",
    "x13": "local_cheese_1kg_usd",
    "x14": "chicken_fillets_1kg_usd",
    "x15": "beef_1kg_usd",
    "x16": "apples_1kg_usd",
    "x17": "banana_1kg_usd",
    "x18": "oranges_1kg_usd",
    "x19": "tomato_1kg_usd",
    "x20": "potato_1kg_usd",
    "x21": "onion_1kg_usd",
    "x22": "lettuce_usd",
    "x23": "water_bottle_15l_usd",
    "x24": "wine_midrange_usd",
    "x25": "domestic_beer_supermarket_usd",
    "x26": "imported_beer_supermarket_usd",
    "x27": "cigarettes_20pack_usd",
    # Transport
    "x28": "transit_ticket_oneway_usd",
    "x29": "transit_monthly_pass_usd",
    "x30": "taxi_start_usd",
    "x31": "taxi_1km_usd",
    "x32": "taxi_1hr_wait_usd",
    "x33": "gasoline_1l_usd",
    "x34": "new_car_vw_golf_usd",
    "x35": "new_car_toyota_corolla_usd",
    # Utilities
    "x36": "utilities_monthly_usd",
    "x37": "mobile_plan_monthly_usd",
    "x38": "internet_monthly_usd",
    # Sports & leisure
    "x39": "gym_membership_monthly_usd",
    "x40": "tennis_court_1hr_usd",
    "x41": "cinema_ticket_usd",
    # Education
    "x42": "preschool_monthly_usd",
    "x43": "intl_primary_school_annual_usd",
    # Clothing
    "x44": "jeans_usd",
    "x45": "summer_dress_usd",
    "x46": "leather_shoes_usd",
    "x47": "running_shoes_usd",
    # Rent
    "x48": "rent_1br_city_centre_usd",
    "x49": "rent_1br_outside_centre_usd",
    "x50": "rent_3br_city_centre_usd",
    "x51": "rent_3br_outside_centre_usd",
    # Property
    "x52": "property_price_m2_centre_usd",
    "x53": "property_price_m2_outside_usd",
    # Finance
    "x54": "col_salary_net_usd",   # prefixed 'col_' to distinguish from net_salary
    "x55": "mortgage_rate_pct",
}

# Also accept human-readable aliases as keys
_ALIAS_TO_CODE = {v: k for k, v in COL_CATALOGUE.items()}

def resolve_columns(user_list: list[str]) -> list[tuple[str, str]]:
    """
    Resolve user list (mix of x-codes and aliases) to
    [(x_code, output_col_name), ...]
    """
    resolved = []
    for item in user_list:
        code = item if item in COL_CATALOGUE else _ALIAS_TO_CODE.get(item)
        if code is None:
            print(f"  ⚠️  Unknown column '{item}' — skipped")
            continue
        resolved.append((code, COL_CATALOGUE[code]))
    return resolved


resolved = resolve_columns(COLUMNS_TO_ADD)
print(f"✅ Will add {len(resolved)} column(s) from the cost-of-living dataset:")
for code, label in resolved:
    print(f"   {code:5s} → {label}")

✅ Will add 1 column(s) from the cost-of-living dataset:
   x3    → mcmeal_combo_usd


In [19]:
# ── 6.2  Load & prepare the big cost-of-living dataset ───────────────────────
col_raw = pd.read_csv(DATA_DIR / "cost_of_living" / "cost-of-living_v2.csv")

col_raw = col_raw.rename(columns={"city": "city", "country": "country"})
col_raw["city"]    = col_raw["city"].astype(str).str.strip()
col_raw["country"] = col_raw["country"].astype(str).str.strip()

# Keep only the x-columns we actually need + identifiers
codes_needed = [code for code, _ in resolved]
col_slim = col_raw[["city", "country"] + codes_needed].copy()

# Rename x-codes to human labels right away
col_slim = col_slim.rename(columns={code: label for code, label in resolved})

# Numeric coercion
for _, label in resolved:
    col_slim[label] = pd.to_numeric(col_slim[label], errors="coerce")

# Some cities appear multiple times (duplicate entries in Numbeo scrape).
# Aggregate by taking the mean of numeric columns after grouping by city+country.
label_cols = [label for _, label in resolved]
col_slim = (
    col_slim
    .groupby(["city", "country"], as_index=False)[label_cols]
    .mean(numeric_only=True)
)

col_slim = add_norm_keys(col_slim, "city", "country")

print(f"Cost-of-living slim: {col_slim.shape[0]} cities × {col_slim.shape[1]} cols")
display(col_slim.head(3))

Cost-of-living slim: 4956 cities × 5 cols


,city,country,mcmeal_combo_usd,_city_key,_country_key
0,'s-Hertogenbosch,Netherlands,9.11,'s,netherlands
1,A Coruna,Spain,7.38,a coruna,spain
2,Aachen,Germany,8.43,aachen,germany


In [20]:
# ── 6.3  Fuzzy-join cost-of-living columns onto base dataset ──────────────────
base_enriched = add_norm_keys(base, "city", "country")

col_keys     = col_slim["_city_key"].tolist()
col_ctry_keys = col_slim["_country_key"].tolist()

# Pre-build result arrays
result_rows = {label: [] for _, label in resolved}
col_match_scores = []

for _, row in base_enriched.iterrows():
    lk = row["_city_key"]
    lc = row["_country_key"]

    if not lk:
        for _, label in resolved:
            result_rows[label].append(np.nan)
        col_match_scores.append(np.nan)
        continue

    res = process.extractOne(lk, col_keys, scorer=FUZZY_SCORER,
                             score_cutoff=FUZZY_THRESHOLD)
    if res is None:
        for _, label in resolved:
            result_rows[label].append(np.nan)
        col_match_scores.append(np.nan)
        continue

    _, score, pos = res
    matched_col_row = col_slim.iloc[pos]

    # Country guard
    rc = col_ctry_keys[pos]
    if lc and rc and lc != rc:
        for _, label in resolved:
            result_rows[label].append(np.nan)
        col_match_scores.append(np.nan)
        continue

    for _, label in resolved:
        result_rows[label].append(matched_col_row[label])
    col_match_scores.append(score)

for label, values in result_rows.items():
    base_enriched[label] = values
base_enriched["_col_match_score"] = col_match_scores

# Drop internal keys
base_enriched = base_enriched.drop(columns=["_city_key", "_country_key"], errors="ignore")

matched_col = (base_enriched["_col_match_score"].notna()).sum()
print(f"\n✅ Cost-of-living enrichment: {matched_col}/{len(base_enriched)} cities matched")
for _, label in resolved:
    n = base_enriched[label].notna().sum()
    print(f"   {label}: {n} non-null values")

display(base_enriched[["city", "country"] + [label for _, label in resolved]].dropna(how="all", subset=[label for _, label in resolved]).head(8))


✅ Cost-of-living enrichment: 202/327 cities matched
   mcmeal_combo_usd: 202 non-null values


,city,country,mcmeal_combo_usd
0,Aarhus,Denmark,12.47
2,Adelaide,Australia,8.65
3,Ahmedabad,India,3.68
5,Almaty,Kazakhstan,5.31
6,Amman,Jordan,7.05
7,Amsterdam,Netherlands,10.54
9,Andorra,Andorra,8.43
10,Ankara,Turkey,3.89


## 7. Save output

In [21]:
out_dir = Path("data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

# ── (a) Base QoL + salary (no cost-of-living enrichment) ──────────────────────
base_path = out_dir / "city_qol_base.csv"
base.to_csv(base_path, index=False)
print(f"💾 Base dataset saved   : {base_path}  ({base.shape[0]} rows × {base.shape[1]} cols)")

# ── (b) Enriched dataset (includes selected cost-of-living columns) ────────────
enriched_path = out_dir / "city_qol_enriched.csv"
base_enriched.to_csv(enriched_path, index=False)
print(f"💾 Enriched dataset saved: {enriched_path}  ({base_enriched.shape[0]} rows × {base_enriched.shape[1]} cols)")

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n📋 Column inventory (enriched):")
for col in base_enriched.columns:
    n_null = base_enriched[col].isna().sum()
    pct    = 100 * n_null / len(base_enriched)
    print(f"   {col:<45s}  {n_null:>3} NaN  ({pct:4.1f}%)")

💾 Base dataset saved   : data/processed/city_qol_base.csv  (327 rows × 33 cols)
💾 Enriched dataset saved: data/processed/city_qol_enriched.csv  (327 rows × 35 cols)

📋 Column inventory (enriched):
   city                                             0 NaN  ( 0.0%)
   country                                          0 NaN  ( 0.0%)
   continent                                       61 NaN  (18.7%)
   qol_index_numbeo                               177 NaN  (54.1%)
   purchasing_power_idx_numbeo                    177 NaN  (54.1%)
   safety_idx_numbeo                              177 NaN  (54.1%)
   healthcare_idx_numbeo                          177 NaN  (54.1%)
   cost_of_living_idx_numbeo                      177 NaN  (54.1%)
   property_price_to_income_numbeo                177 NaN  (54.1%)
   traffic_commute_idx_numbeo                     177 NaN  (54.1%)
   pollution_idx_numbeo                           177 NaN  (54.1%)
   climate_idx_numbeo                             177 NaN  (54.1%)